Pre-analysis plots used in the publication

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator
plt.rcParams.update({'font.size': 20, 'font.family': 'Arial'})
# ---- Data digitized from panel A (approximate, read off the log-scale axis) ----
labels = ["\nNC", "\nGain", "\nLoss", "\nGain\n(de novo)"]

points = {
    "NC":             [1.6e9, 2.0e9, 2.6e9],
    "Gain":           [5.0e3, 1.0e4, 1.0e5],
    "Loss":           [1.5e5, 2.5e5, 5.0e5],
    "Gain (de novo)": [2.5e3, 5.0e3, 1.5e4],
}

# colors follow the DVG classification scheme
colors = ["white", "#00c1c4", "#f2c811", "#ec2d8a"]   # NC, gain, loss, de novo gain

means = [10 ** np.mean(np.log10(v)) for v in points.values()]
sems = [np.std(np.log10(v), ddof=1) / np.sqrt(len(v)) for v in points.values()]
lower = [m - 10 ** (np.log10(m) - s) for m, s in zip(means, sems)]
upper = [10 ** (np.log10(m) + s) - m for m, s in zip(means, sems)]

x = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(6,3.5), dpi=800)

ax.bar(x, means, width=0.6, color=colors, edgecolor="black", lw=2, zorder=2)
ax.errorbar(x, means, yerr=[lower, upper], fmt="none",
            ecolor="black", capsize=6, lw=3, zorder=3)

rng = np.random.default_rng(0)
for i, vals in enumerate(points.values()):
    jitter = rng.uniform(-0.12, 0.12, size=len(vals))
    ax.scatter(x[i] + jitter, vals, s=24, color="#3a3a3a",
               edgecolors="none", zorder=4)

ax.set_yscale("log")
ax.set_ylim(1e2, 1e10)

# y-axis label pushed further from the axis
ax.set_ylabel("Virus concentration (PFU/mL)", labelpad=10, y=-0.3, ha="left")

# major ticks every 2 decades, with the top limit itself carrying a tick+label
ax.set_yticks(10.0 ** np.arange(2, 11, 2))
ax.yaxis.set_minor_locator(LogLocator(base=10, subs="auto", numticks=100))

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlim(-0.6, len(labels) - 0.4)
ax.tick_params(direction="out")
ax.tick_params(axis="x", pad=6)   # x tick labels a bit further from the axis


def sig_bracket(ax, i, j, y, text, h=0.12):
    """Significance bracket between categories i and j, at log10-height y."""
    ax.plot([i, i, j, j], [10 ** y, 10 ** (y + h), 10 ** (y + h), 10 ** y],
            lw=2, c="black", zorder=5)
    ax.text((i + j) / 2, 10 ** (y + h + 0.05), text,
            ha="center", va="bottom")


sig_bracket(ax, 1, 2, 6.3, "*")   # Gain vs Loss
sig_bracket(ax, 2, 3, 5.9, "*")   # Loss vs Gain (de novo)



In [ ]:
from utils.dip_utils import *
from utils.dip_visuals import *

import matplotlib.pyplot as plt
import pelz_datasets
import numpy as np

from statsmodels.tsa.stattools import adfuller

import warnings
warnings.filterwarnings('ignore')


In [ ]:

long_pelz_readcounts_abs = pelz_datasets.long_readcounts(cutoff=15)
long_pelz_readcounts_abs = long_pelz_readcounts_abs.dropna()

measure_ids = long_pelz_readcounts_abs.columns[4:]
long_pelz_readcounts_abs

cultivation_values = pelz_datasets.pelz_cultivation_values()
cultivation_values.drop(columns=['key', 'unit'], inplace=True)
cultivation_values
tp2dpi = cultivation_values.T['dpi'].apply(lambda d : round(d,2)).to_dict()

read_counts_T = long_pelz_readcounts_abs.copy()
read_counts_T.index = read_counts_T.key
read_counts_T = read_counts_T[long_pelz_readcounts_abs.columns[4:]].T
read_counts_T.index = [round(tp2dpi[idx], 2) for idx in read_counts_T.index]
read_counts_T

cult_df = cultivation_values.T[['dpi', 'plaque_assay']]
cult_df.index = [round(d,2) for d in cult_df.dpi]
cult_df = cult_df.drop(columns=['dpi'])

ts_df = pd.merge(left=cult_df,
              right=read_counts_T,
              left_index=True,
              right_index=True,
              how='left')
ts_df['plaque_assay'] = pd.to_numeric(ts_df['plaque_assay'], errors='coerce')
ts_df


In [ ]:
pfu_plot_df = ts_df[['plaque_assay']].copy().dropna()
dvg_plot_df = ts_df.drop(columns=['plaque_assay']).copy().dropna()
start = dvg_plot_df.iloc[0]
end   = dvg_plot_df.iloc[-1]

# first non-zero value per column (vectorised)
first_nz = dvg_plot_df.replace(0, np.nan).bfill().iloc[0]

de_novo = (start == 0) & (end > 0)
gain    = (start >  0) & (end > start)
loss    = (start >  0) & (end < start)

diff = pd.Series(np.nan, index=dvg_plot_df.columns)
diff[de_novo] = end[de_novo] - first_nz[de_novo]   # non-zero start -> end
diff[gain]    = end[gain]    - start[gain]
diff[loss]    = start[loss]  - end[loss]

top_idx = {
    'de_novo_gain': diff[de_novo].nlargest(5).index,
    'gain':         diff[gain].nlargest(5).index,
    'loss':         diff[loss].nlargest(5).index,
}

top_dvg_df = dvg_plot_df[[c for idx in top_idx.values() for c in idx]]

# category lookup, handy for colouring/faceting later
dvg_category = pd.Series(
    {c: cat for cat, idx in top_idx.items() for c in idx},
    name='category'
)

In [ ]:

fig, axs = plt.subplots(figsize=(14,8), dpi=800, nrows=2)
ax1, ax2 = axs

ax2.set_xlabel('Time post infection (days)')
ax2.set_xticks([range(0, max(ts_df.index.astype(int))+1)[i] for i in range(0, max(ts_df.index.astype(int))+1, 1)])
ax1.set_xticks([range(0, max(ts_df.index.astype(int))+1)[i] for i in range(0, max(ts_df.index.astype(int))+1, 1)])

ax2.set_xlim(-1, max(ts_df.index.astype(int))+1)
ax1.set_xlim(-1, max(ts_df.index.astype(int))+1)

ax1.plot(pfu_plot_df.index, pfu_plot_df['plaque_assay'], 
        color='black', 
        marker='D',
        linewidth=2,
        markersize=8,
        label='Infectious virus\nconcentration (PFU/mL)'
        )
ax1.set_ylabel('Plaque forming units (PFU/mL)')
ax1.set_yscale('log')

ax1.legend(loc='center left', bbox_to_anchor=(1, 0.5))

cmap = plt.get_cmap('Greys')
ax2.set_ylabel('DVG read counts (abs)')
ax2.set_yscale('log')
ax2.set_prop_cycle(color=cmap(np.linspace(0.2, 0.8, dvg_plot_df.shape[1])))
ax2.plot(dvg_plot_df.index, dvg_plot_df.values,
        alpha=0.1,
        marker='.',
        linewidth=1
        )

cat_colors = {'loss': "#f2c811", 'gain': "#00c1c4", 'de_novo_gain': "#ec2d8a"}
cat_markers = {'de_novo_gain': 'D', 'gain': 's', 'loss': '^'}
for cat, color in cat_colors.items():
    cols = dvg_category.index[dvg_category == cat]
    ax2.plot(top_dvg_df.index, top_dvg_df[cols].replace(0, np.nan),
            color=color,
            alpha=1,
            marker=cat_markers[cat],
            linewidth=1.5
            )

from matplotlib.lines import Line2D
cat_labels = {
    'loss': 'Top loss',
    'gain': 'Top gain',
    'de_novo_gain': 'Top $\\it{de\\ novo}$ gain',
}
handles = [Line2D([], [], color='grey', marker='.', lw=1, alpha=0.5, label='All DVGs')]
handles += [Line2D([], [], color=cat_colors[c], marker=cat_markers[c], lw=1.5,
                    label=cat_labels[c])
            for c in cat_colors]
ax2.legend(handles=handles, loc='center left', bbox_to_anchor=(1, 0.5))

plt.show()

In [ ]:
dvg_plot_df.sort_values(by=dvg_plot_df.index[-1], axis=1, ascending=False).head(10)

In [ ]:
plt.rcParams.update({'font.size': 18, 'font.family': 'Arial'})

fig, axs = plt.subplots(figsize=(5,6), dpi=800, nrows=2)
ax1, ax2 = axs

ax2.set_xlabel('Time post infection (days)')
ax2.set_xticks([range(0, max(ts_df.index.astype(int))+1)[i] for i in range(0, max(ts_df.index.astype(int))+1, 3)])
ax1.set_xticks([range(0, max(ts_df.index.astype(int))+1)[i] for i in range(0, max(ts_df.index.astype(int))+1, 3)])

ax2.set_xlim(-1, max(ts_df.index.astype(int))+1)
ax1.set_xlim(-1, max(ts_df.index.astype(int))+1)

ax1.plot(pfu_plot_df.index, pfu_plot_df['plaque_assay'], 
        color='black', 
        marker='D',
        linewidth=1.5,
        markersize=5,
        label='Infectious\nvirus titer'
        )
ax1.set_ylabel('Infectious virus\n(PFU/mL)', y=0.5)
ax1.set_yscale('log')

ax1.legend(loc='center left', bbox_to_anchor=(1, 0.5))

cmap = plt.get_cmap('Greys')
ax2.set_ylabel('DVG read counts (abs)')
ax2.set_yscale('log')
ax2.set_prop_cycle(color=cmap(np.linspace(0.2, 0.8, dvg_plot_df.shape[1])))
ax2.plot(dvg_plot_df.index, dvg_plot_df.replace(0, np.nan),
        alpha=0.1,
        marker='.',
        linewidth=1,
        markeredgewidth=0.5,
        markeredgecolor='white'
        )

candidates = {
    'PB2_129_2176': 'loss',
    'PB2_217_2204': 'gain',
    'PB2_269_2202': 'de_novo_gain',
}

cat_colors = {'loss': "#f2c811", 'gain': "#00c1c4", 'de_novo_gain': "#ec2d8a"}
cat_markers = {'de_novo_gain': 'D', 'gain': 's', 'loss': '^'}

for col, cat in candidates.items():
    ax2.plot(dvg_plot_df.index, dvg_plot_df[col].replace(0, np.nan),
            color=cat_colors[cat],
            alpha=1,
            marker=cat_markers[cat],
            linewidth=2,
            markersize=5,
            markeredgewidth=0.5,
            markeredgecolor='white'
            )

from matplotlib.lines import Line2D
cat_labels = {
    'loss': 'loss',
    'gain': 'gain',
    'de_novo_gain': '$\\it{de\\ novo}$ gain',
}
handles = [Line2D([], [], color='grey', marker='.', lw=1, alpha=0.5, label='All DVGs')]
handles += [Line2D([], [], color=cat_colors[cat], marker=cat_markers[cat], lw=2, markeredgewidth=1.5, markeredgecolor='white',
                    label=f'{cat_labels[cat]}')
            for col, cat in candidates.items()]
ax2.legend(handles=handles, loc='center left', bbox_to_anchor=(1, 0.5))

plt.show()

In [ ]:
plt.rcParams.update({'font.size': 18, 'font.family': 'Arial'})

fig1, ax1 = plt.subplots(figsize=(7,3), dpi=800)

ax1.set_xlabel('Time post infection (days)')
ax1.set_xticks([range(0, max(ts_df.index.astype(int))+1)[i] for i in range(0, max(ts_df.index.astype(int))+1, 3)])
ax1.set_xlim(-1, max(ts_df.index.astype(int))+1)

ax1.plot(pfu_plot_df.index, pfu_plot_df['plaque_assay'], 
        color='black', 
        marker='D',
        linewidth=1.5,
        markersize=5,
        label='Infectious\nvirus titer'
        )
ax1.set_ylabel('Infectious virus\n(PFU/mL)', y=0.5)
ax1.set_yscale('log')

ax1.legend(loc='center left', bbox_to_anchor=(1, 0.5))

plt.show()


fig2, ax2 = plt.subplots(figsize=(7,3.3), dpi=800)

ax2.set_xlabel('Time post infection (days)')
ax2.set_xticks([range(0, max(ts_df.index.astype(int))+1)[i] for i in range(0, max(ts_df.index.astype(int))+1, 3)])
ax2.set_xlim(-1, max(ts_df.index.astype(int))+1)

cmap = plt.get_cmap('Greys')
ax2.set_ylabel('DVG read counts (abs)')
ax2.set_yscale('log')
ax2.set_prop_cycle(color=cmap(np.linspace(0.2, 0.8, dvg_plot_df.shape[1])))
ax2.plot(dvg_plot_df.index, dvg_plot_df.replace(0, np.nan),
        alpha=0.1,
        marker='.',
        linewidth=1,
        markeredgewidth=0.5,
        markeredgecolor='white'
        )

candidates = {
    'PB2_129_2176': 'loss',
    'PB2_217_2204': 'gain',
    'PB2_269_2202': 'de_novo_gain',
}

cat_colors = {'loss': "#f2c811", 'gain': "#00c1c4", 'de_novo_gain': "#ec2d8a"}
cat_markers = {'de_novo_gain': 'D', 'gain': 's', 'loss': '^'}

for col, cat in candidates.items():
    ax2.plot(dvg_plot_df.index, dvg_plot_df[col].replace(0, np.nan),
            color=cat_colors[cat],
            alpha=1,
            marker=cat_markers[cat],
            linewidth=2,
            markersize=5,
            markeredgewidth=0.5,
            markeredgecolor='white'
            )

from matplotlib.lines import Line2D
cat_labels = {
    'loss': 'loss',
    'gain': 'gain',
    'de_novo_gain': '$\\it{de\\ novo}$ gain',
}
handles = [Line2D([], [], color='grey', marker='.', lw=1, alpha=0.5, label='All DVGs')]
handles += [Line2D([], [], color=cat_colors[cat], marker=cat_markers[cat], lw=2, markeredgewidth=1.5, markeredgecolor='white',
                    label=f'{cat_labels[cat]}')
            for col, cat in candidates.items()]
ax2.legend(handles=handles, loc='center left', bbox_to_anchor=(1, 0.5))

plt.show()